# 11a — Batch Normalization and Dropout

Corresponds to: Note 11 — Batch Normalization and Dropout

**Libraries introduced in this notebook:** No new libraries — numpy only, introduced in Note 03.

**What this notebook demonstrates:**
1. Batch norm forward pass — four steps implemented from scratch, worked example verified against Note 11 Part 2
2. Covariate shift — activation distributions across layers with and without batch norm
3. Dropout — inverted dropout implemented from scratch, training vs inference mode comparison

In [ ]:
import numpy as np

## Part 1: Batch Norm Forward Pass

Batch normalization applies four steps to a mini-batch of pre-activations z:

    Step 1:  μ_B  = (1/m) Σ zᵢ
    Step 2:  σ²_B = (1/m) Σ (zᵢ − μ_B)²
    Step 3:  ẑᵢ   = (zᵢ − μ_B) / √(σ²_B + ε)
    Step 4:  yᵢ   = γ · ẑᵢ + β

We implement each step explicitly and reproduce the worked example from Note 11 Part 2:
z = [2.0, 4.0, 6.0, 8.0], γ = 2.0, β = 1.0, ε = 0.

In [ ]:
z     = np.array([2.0, 4.0, 6.0, 8.0])
gamma = 2.0
beta  = 1.0
eps   = 0.0

# Step 1 — batch mean
mu_B = z.mean()
print(f"Step 1 — batch mean:      μ_B  = {mu_B}")

# Step 2 — batch variance
var_B = ((z - mu_B) ** 2).mean()
print(f"Step 2 — batch variance:  σ²_B = {var_B}")

# Step 3 — normalize
z_hat = (z - mu_B) / np.sqrt(var_B + eps)
print(f"Step 3 — normalized:      ẑ    = {np.round(z_hat, 3)}")
print(f"         verify mean(ẑ) = {z_hat.mean():.6f}  (expected 0.0)")

# Step 4 — scale and shift
y = gamma * z_hat + beta
print(f"Step 4 — scale and shift: y    = {np.round(y, 3)}")
print(f"         output mean = {y.mean():.3f}  (expected β = {beta})")
print(f"         output std  = {y.std():.3f}   (expected γ = {gamma})")

The outputs match the hand calculation in Note 11 Part 2 exactly.

Now wrap the four steps into a reusable function that also returns the intermediate values needed for backpropagation.

In [ ]:
def batch_norm_forward(z, gamma, beta, eps=1e-8):
    mu_B  = z.mean(axis=0)
    var_B = z.var(axis=0)
    z_hat = (z - mu_B) / np.sqrt(var_B + eps)
    y     = gamma * z_hat + beta
    cache = (z, z_hat, mu_B, var_B, gamma, eps)
    return y, cache

# Verify on the same example (eps=1e-8 this time)
y_out, _ = batch_norm_forward(z, gamma=2.0, beta=1.0)
print(f"batch_norm_forward output: {np.round(y_out, 3)}")
print(f"mean = {y_out.mean():.3f},  std = {y_out.std():.3f}")

## Part 2: Covariate Shift Demonstration

Internal covariate shift means activation distributions drift across layers during a forward pass.
We demonstrate this by running 1000 random inputs through a 5-layer ReLU network and recording
the mean and std of activations at each layer — once without batch norm, once with.

Without batch norm, the distribution shifts at every layer.
With batch norm, mean and std are stabilised at each layer.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def forward_pass(x, weights, use_bn):
    """Forward pass through a list of weight matrices. Returns (mean, std) per layer."""
    a = x
    stats = []
    for W in weights:
        z = a @ W
        if use_bn:
            mu  = z.mean(axis=0)
            std = z.std(axis=0) + 1e-8
            z   = (z - mu) / std          # normalize only — γ=1, β=0 for this demo
        a = relu(z)
        stats.append((a.mean(), a.std()))
    return stats

rng        = np.random.default_rng(42)
n_units    = 128
n_layers   = 5
n_samples  = 1000

x       = rng.standard_normal((n_samples, n_units))
weights = [rng.normal(0, np.sqrt(2 / n_units), (n_units, n_units)) for _ in range(n_layers)]

stats_no_bn = forward_pass(x, weights, use_bn=False)
stats_bn    = forward_pass(x, weights, use_bn=True)

print(f"{'Layer':<8} {'No BN mean':>12} {'No BN std':>12} {'BN mean':>12} {'BN std':>12}")
print("-" * 60)
for l, (no_bn, bn) in enumerate(zip(stats_no_bn, stats_bn), 1):
    print(f"{l:<8} {no_bn[0]:>12.4f} {no_bn[1]:>12.4f} {bn[0]:>12.4f} {bn[1]:>12.4f}")

Without batch norm, mean and std drift across layers — each layer receives a different distribution.
With batch norm, mean stays near 0 and std stays near 0.5 at every layer (the post-ReLU value of a
normalized distribution), confirming that batch norm stabilises the input distribution to each layer.

## Part 3: Dropout — Training vs Inference

Dropout was introduced in Note 07. Here we implement inverted dropout from scratch and demonstrate
the training/inference mode difference explicitly.

Inverted dropout:
- Training:   zero each neuron with probability p, scale survivors by 1/(1−p)
- Inference:  all neurons active, no scaling — expected magnitude is already correct

We use a single hidden layer of 8 neurons so the mask is easy to inspect.

In [ ]:
def dropout(a, p, training, rng):
    """Inverted dropout. p = drop probability."""
    if not training:
        return a, None
    mask  = (rng.random(a.shape) > p).astype(float)
    a_out = a * mask / (1 - p)
    return a_out, mask

rng = np.random.default_rng(0)
a   = np.array([0.5, 0.8, 0.3, 0.9, 0.1, 0.7, 0.4, 0.6])   # 8 neurons
p   = 0.5

# Training mode — run three steps to show different masks each time
print("Training mode (p=0.5):")
print(f"  Input activations: {a}")
for step in range(3):
    a_out, mask = dropout(a, p, training=True, rng=rng)
    print(f"  Step {step+1} mask:   {mask.astype(int)}")
    print(f"  Step {step+1} output: {np.round(a_out, 3)}")

print()

# Inference mode — deterministic, all neurons active
a_out_eval, mask_eval = dropout(a, p, training=False, rng=rng)
print("Inference mode (p=0.5):")
print(f"  Output: {a_out_eval}")
print(f"  Identical to input: {np.allclose(a_out_eval, a)}")

Each training step uses a different random mask — no neuron is guaranteed to be present.
At inference, all neurons are active and the output equals the input exactly.

Now verify that inverted dropout preserves expected activation magnitude across many steps.

In [ ]:
rng    = np.random.default_rng(1)
a_large = rng.standard_normal(10000)   # large vector to get stable estimates
p       = 0.5

a_dropped, _ = dropout(a_large, p, training=True, rng=rng)

print(f"Original mean:        {a_large.mean():.4f}")
print(f"Original std:         {a_large.std():.4f}")
print(f"After dropout mean:   {a_dropped.mean():.4f}  (expected ≈ original)")
print(f"After dropout std:    {a_dropped.std():.4f}  (expected ≈ original)")
print()
print("Inverted dropout preserves expected magnitude — no rescaling needed at inference.")

## Summary

This notebook demonstrated:

1. **Batch norm forward pass** — four steps implemented explicitly. Worked example z = [2, 4, 6, 8] with γ=2, β=1 reproduces the hand calculation in Note 11 Part 2 exactly. Output mean equals β and output std equals γ.
2. **Covariate shift** — without batch norm, activation mean and std drift across 5 layers. With batch norm, they remain stable at every layer, confirming that batch norm stabilises the input distribution each layer receives.
3. **Dropout** — inverted dropout implemented from scratch. Training mode applies a different random mask each step and scales survivors by 1/(1−p). Inference mode passes activations unchanged. Expected magnitude is preserved across both modes — no rescaling needed at inference.